In [1]:
import os

In [16]:

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

In [3]:
loader = PyMuPDFLoader('attention-is-all-you-need-Paper.pdf')

In [4]:
document = loader.load()

<h3>Text Splitter</H3>

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap=150
)

In [7]:
chunks = text_splitter.split_documents(document)

In [8]:
print('the length of chuks is: ',len(chunks))

the length of chuks is:  53


<h3>Embedding</h3>

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<h3>Vector database</h3>

In [11]:
vector_database = FAISS.from_documents(
    chunks,
    embedding
)

<H3>Retriver</H3>

In [12]:
retriver = vector_database.as_retriever(
    search_kwargs = {'k':3}
)

<h3>Create the model</h3>

In [13]:
llm_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash',
    temperature=0
)

<h3>Prompt Template</h3>

In [17]:
prompt= ChatPromptTemplate.from_template(
    '''
    Answer question from the context.
    if a questions is ou of context just say:"Out of context question"    
    
    Context
    {context}

    Question
    {question}
'''
)

<h3>Question</h3>

In [18]:
question = input('enter your question: ')

<h3>retirve the question</h3>

In [19]:
retrived_document = retriver.invoke(question)

<h3>Context</h3>

In [21]:
context = '\n\n'.join(
    pages.page_content
    for pages in retrived_document
)

In [23]:
final_prompt = prompt.invoke({
    'context':context,
    'question':question
})

<h3>ENAGAE THE MODEL</h3>

In [24]:
response = llm_model.invoke(question)

e:\GEN-AI-PROJECTS\genai-env\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [25]:
print(response.text())

Hello! How can I help you today? 

You can ask me a question, request help with writing, coding, brainstorming, summarizing text, or solving a problem. What's on your mind?


e:\GEN-AI-PROJECTS\genai-env\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)
